# Notebook 03 - Richer features

Notebook 02 gave a baseline: a random forest on amino-acid fractions, charge and hydropathy scored ROC AUC 0.933 and MCC 0.715 on cluster-split test peptides, with the leading methionine removed. In this notebook I add richer sequence features and check whether they beat that baseline on the same test peptides.

First I load the cleaned data and look at its columns, so I can rebuild the same train and test split.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/peptides_clean.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(9558, 6)
['uid', 'id', 'seq', 'label', 'length', 'cluster']


,uid,id,seq,label,length,cluster
0,s0,DRAMP29098,GLSRLFTALK,1,10,16
1,s1,DRAMP30027,EEQAKTFLDK,1,10,102
2,s2,DRAMP32992,LKKLKFLKKQ,1,10,26
3,s3,DRAMP31465,FFVIFYRRKK,1,10,42
4,s4,DRAMP35956,FHAVPQSFYT,1,10,4


In [2]:
split = pd.read_csv("../data/processed/split.csv")
df = df.merge(split, on="uid")
print(df["split"].value_counts())
print(df.groupby("split")["cluster"].nunique())

split
train    7688
test     1870
Name: count, dtype: int64
split
test      290
train    1160
Name: cluster, dtype: int64


## Part 1 - Check the split and remove the leading M
The split from notebook 02 has 7688 training and 1870 test peptides. I first check that no cluster appears in both sets, then remove the leading methionine from every sequence, as in my cleaner baseline.

In [3]:
overlap = set(df[df["split"] == "train"]["cluster"]) & set(df[df["split"] == "test"]["cluster"])
print("Clusters in both train and test:", len(overlap))

df["seq"] = df["seq"].apply(lambda s: s[1:] if s.startswith("M") else s)
print(df["seq"].head(3).tolist())

Clusters in both train and test: 0
['GLSRLFTALK', 'EEQAKTFLDK', 'LKKLKFLKKQ']


## Part 2 - Build richer features
The baseline used amino-acid fractions, net charge and mean hydropathy. Here I add:
- physicochemical properties: charge at pH 7.4, isoelectric point, aromaticity, instability index and hydrophobic moment (how strongly hydrophobic residues sit on one side of an alpha helix, a known feature of AMPs)
- group fractions: hydrophobic, polar, positive, negative and small residues
- dipeptide composition: how often each pair of neighbouring amino acids occurs (400 features)

Length is left out on purpose, because both classes have the same length distribution.

In [4]:
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis

AA = "ACDEFGHIKLMNPQRSTVWY"
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5,
      "G": -0.4, "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8,
      "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
GROUPS = {"hydrophobic": "AVILMFWC", "polar": "STNQ", "positive": "KR",
          "negative": "DE", "small": "GAS"}

def hydrophobic_moment(seq, angle=100):
    theta = np.radians(angle) * np.arange(len(seq))
    h = np.array([KD[a] for a in seq])
    return np.hypot((h * np.cos(theta)).sum(), (h * np.sin(theta)).sum()) / len(seq)

def rich_features(seq):
    n = len(seq)
    pa = ProteinAnalysis(seq)
    f = {f"frac_{a}": seq.count(a) / n for a in AA}
    f["charge_pH7"] = pa.charge_at_pH(7.4)
    f["pI"] = pa.isoelectric_point()
    f["aromaticity"] = pa.aromaticity()
    f["instability"] = pa.instability_index()
    f["mean_hydropathy"] = pa.gravy()
    f["hydrophobic_moment"] = hydrophobic_moment(seq)
    for g, letters in GROUPS.items():
        f[f"grp_{g}"] = sum(seq.count(a) for a in letters) / n
    for a in AA:
        for b in AA:
            f[f"dp_{a}{b}"] = sum(seq[i:i+2] == a + b for i in range(n - 1)) / (n - 1)
    return f

bad = df[~df["seq"].apply(lambda s: set(s) <= set(AA))]
print("Sequences with non-standard letters:", len(bad))

X_all = pd.DataFrame([rich_features(s) for s in df["seq"]])
print(X_all.shape)
X_all.iloc[:3, [20, 21, 22, 23, 24, 25]]

Sequences with non-standard letters: 0
(9558, 431)


,charge_pH7,pI,aromaticity,instability,mean_hydropathy,hydrophobic_moment
0,1.554922,11.000837,0.1,19.77,0.57,1.660016
1,-1.336043,4.679463,0.1,64.71,-1.41,1.458756
2,4.544925,10.602487,0.1,-1.32,-0.88,1.249208


## Part 3 - Random forest with the richer features
I train the same random forest settings as the baseline (300 trees, seed 42) on all 431 features, using the same training peptides. Only the features change, so any difference in score comes from the features. I score the test set once and do not tune anything on it.

The baseline to beat, with the leading M removed: accuracy 0.857, F1 0.854, ROC AUC 0.933, MCC 0.715.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef

is_train = (df["split"] == "train").values
is_test = (df["split"] == "test").values
y_train, y_test = df.loc[is_train, "label"], df.loc[is_test, "label"]

rf_rich = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_rich.fit(X_all[is_train], y_train)
pred = rf_rich.predict(X_all[is_test])
prob = rf_rich.predict_proba(X_all[is_test])[:, 1]

baseline = pd.Series({"accuracy": 0.857, "F1": 0.854, "ROC AUC": 0.933, "MCC": 0.715})
rich = pd.Series({"accuracy": accuracy_score(y_test, pred), "F1": f1_score(y_test, pred),
                  "ROC AUC": roc_auc_score(y_test, prob), "MCC": matthews_corrcoef(y_test, pred)})
print(pd.DataFrame({"baseline": baseline, "richer features": rich}).round(3))
print()
print("Top 10 features:")
print(pd.Series(rf_rich.feature_importances_, index=X_all.columns).sort_values(ascending=False).head(10).round(3))

          baseline  richer features
accuracy     0.857            0.851
F1           0.854            0.844
ROC AUC      0.933            0.931
MCC          0.715            0.706

Top 10 features:
charge_pH7            0.045
hydrophobic_moment    0.034
pI                    0.031
grp_positive          0.026
frac_K                0.021
grp_polar             0.019
grp_negative          0.018
grp_hydrophobic       0.016
frac_C                0.015
frac_P                0.014
dtype: float64


In [6]:
from sklearn.model_selection import GroupKFold, cross_val_score

groups = df.loc[is_train, "cluster"]
X_tr = X_all[is_train]

dp_cols = [c for c in X_all.columns if c.startswith("dp_")]
sets = {
    "simple": [c for c in X_all.columns if c.startswith("frac_")] + ["charge_pH7", "mean_hydropathy"],
    "compact": [c for c in X_all.columns if c not in dp_cols],
    "full": list(X_all.columns),
}

rows = {}
for name, cols_ in sets.items():
    rf_cv = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    s = cross_val_score(rf_cv, X_tr[cols_], y_train, groups=groups,
                        cv=GroupKFold(n_splits=5), scoring="roc_auc")
    rows[name] = {"n_features": len(cols_), "mean ROC AUC": s.mean(), "std": s.std()}

print(pd.DataFrame(rows).T.round(3))

         n_features  mean ROC AUC    std
simple         22.0         0.890  0.021
compact        31.0         0.898  0.020
full          431.0         0.903  0.021


## Part 5 - Choosing the feature set
In 5-fold cluster-grouped cross-validation on the training peptides, ROC AUC was 0.890 for the simple set (22 features), 0.898 for the compact set (31) and 0.903 for the full set (431). Each score varies by about 0.02 between folds, so the differences are within noise.

I choose the compact set for the final model: it performs like the full set, but it is faster to compute for a real-time tool and every feature has a clear biological meaning. I score it on the test peptides once for the record, then train the final model on all peptides and save it for the Streamlit app.

In [7]:
import joblib

final_cols = sets["compact"]
rf_final = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_final.fit(X_all.loc[is_train, final_cols], y_train)
pred = rf_final.predict(X_all.loc[is_test, final_cols])
prob = rf_final.predict_proba(X_all.loc[is_test, final_cols])[:, 1]
print({"accuracy": round(accuracy_score(y_test, pred), 3), "F1": round(f1_score(y_test, pred), 3),
       "ROC AUC": round(roc_auc_score(y_test, prob), 3), "MCC": round(matthews_corrcoef(y_test, pred), 3)})

rf_app = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_app.fit(X_all[final_cols], df["label"])
joblib.dump({"model": rf_app, "features": final_cols}, "../models/amp_rf_compact.joblib")
print("Saved ../models/amp_rf_compact.joblib with", len(final_cols), "features")

{'accuracy': 0.859, 'F1': 0.854, 'ROC AUC': 0.931, 'MCC': 0.721}
Saved ../models/amp_rf_compact.joblib with 31 features


## Part 6 - Summary of this notebook
Richer features did not improve on the baseline. On the cluster-split test peptides, the compact model (31 features) scored ROC AUC 0.931 and MCC 0.721, against 0.933 and 0.715 for the simple baseline. Cluster-grouped cross-validation gave the same picture: 0.890 (simple), 0.898 (compact) and 0.903 (full, 431 features), all within fold-to-fold noise.

I chose the compact set because it is fast to compute and each feature is easy to explain. The final model was trained on all 9558 peptides and saved to `models/amp_rf_compact.joblib` for the Streamlit app. The app must remove a leading M from any input sequence, because the model was trained that way.